# 02 — Data Quality & Preprocessing

**Goal:** Clean the data, handle missing values, correct data types, build the preprocessing pipeline, and split into train/test sets.

---

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np

from src.data_preprocessing import (
    load_raw_data, clean_data, validate_dataset,
    separate_features_target, split_data,
    identify_column_types, build_preprocessor, get_feature_names,
    TARGET_COL, ID_COL
)

## 2.1 Load Raw Data

In [2]:
df_raw = load_raw_data()
print(f"Raw shape: {df_raw.shape}")

Loaded WA_Fn-UseC_-Telco-Customer-Churn.csv: 7043 rows, 21 columns
Raw shape: (7043, 21)


## 2.2 Check Duplicates

In [3]:
print(f"Duplicate rows: {df_raw.duplicated().sum()}")
if ID_COL in df_raw.columns:
    print(f"Duplicate customer IDs: {df_raw[ID_COL].duplicated().sum()}")

Duplicate rows: 0
Duplicate customer IDs: 0


## 2.3 Missing and Invalid Values

In [4]:
# Check NaN
print("NaN counts:")
print(df_raw.isnull().sum()[df_raw.isnull().sum() > 0])

# Check blank strings in object columns
print("\nBlank string counts:")
for col in df_raw.select_dtypes(include='object').columns:
    blanks = (df_raw[col].str.strip() == '').sum()
    if blanks > 0:
        print(f"  {col}: {blanks}")

NaN counts:
Series([], dtype: int64)

Blank string counts:
  TotalCharges: 11


## 2.4 Clean Data

In [5]:
df = clean_data(df_raw)
print(f"\nCleaned shape: {df.shape}")
print(f"\nTarget distribution after encoding:")
print(df[TARGET_COL].value_counts())
print(f"\nTotalCharges dtype: {df['TotalCharges'].dtype}")
print(f"TotalCharges NaN count: {df['TotalCharges'].isna().sum()}")
print(f"\nSeniorCitizen dtype: {df['SeniorCitizen'].dtype}")
print(f"SeniorCitizen values: {df['SeniorCitizen'].unique().tolist()}")


Cleaned shape: (7043, 21)

Target distribution after encoding:
Churn
0    5174
1    1869
Name: count, dtype: int64

TotalCharges dtype: float64
TotalCharges NaN count: 11

SeniorCitizen dtype: object
SeniorCitizen values: ['No', 'Yes']


## 2.5 Separate Features and Target

In [6]:
X, y, ids = separate_features_target(df)
print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeature columns: {list(X.columns)}")
print(f"\nTarget NOT in features: {TARGET_COL not in X.columns}")
print(f"ID NOT in features: {ID_COL not in X.columns}")

Features shape: (7043, 19)
Target shape: (7043,)

Feature columns: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges']

Target NOT in features: True
ID NOT in features: True


## 2.6 Train / Test Split

In [7]:
X_train, X_test, y_train, y_test = split_data(X, y)
print(f"\nTrain shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")

Train: 5634 rows | Test: 1409 rows | Train churn rate: 0.265 | Test churn rate: 0.265

Train shape: (5634, 19)
Test shape: (1409, 19)


## 2.7 Build Preprocessing Pipeline

In [8]:
numeric_cols, categorical_cols = identify_column_types(X_train)
print(f"Numeric features ({len(numeric_cols)}): {numeric_cols}")
print(f"Categorical features ({len(categorical_cols)}): {categorical_cols}")

preprocessor = build_preprocessor(numeric_cols, categorical_cols)
print(f"\nPreprocessor: {preprocessor}")

Numeric features (3): ['tenure', 'MonthlyCharges', 'TotalCharges']
Categorical features (16): ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

Preprocessor: ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['tenure', 'MonthlyCharges', 'TotalCharges']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                 

In [9]:
# Fit on training data and transform both sets
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

feature_names = get_feature_names(preprocessor)
print(f"\nTransformed features: {X_train_transformed.shape[1]}")
print(f"\nFeature names (first 20): {feature_names[:20]}")
print(f"\nTrain transformed shape: {X_train_transformed.shape}")
print(f"Test transformed shape: {X_test_transformed.shape}")


Transformed features: 46

Feature names (first 20): ['num__tenure', 'num__MonthlyCharges', 'num__TotalCharges', 'cat__gender_Female', 'cat__gender_Male', 'cat__SeniorCitizen_No', 'cat__SeniorCitizen_Yes', 'cat__Partner_No', 'cat__Partner_Yes', 'cat__Dependents_No', 'cat__Dependents_Yes', 'cat__PhoneService_No', 'cat__PhoneService_Yes', 'cat__MultipleLines_No', 'cat__MultipleLines_No phone service', 'cat__MultipleLines_Yes', 'cat__InternetService_DSL', 'cat__InternetService_Fiber optic', 'cat__InternetService_No', 'cat__OnlineSecurity_No']

Train transformed shape: (5634, 46)
Test transformed shape: (1409, 46)


## 2.8 Verify No Leakage

Checklist:
- [x] Target column excluded from features
- [x] Customer ID excluded from features
- [x] Preprocessor fitted on training data only
- [x] Test data transformed (not fitted) using training preprocessor
- [x] TotalCharges NaN handled by imputer (no manual fill using target info)